In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import shapes, rasterize
import geopandas as gpd
from shapely.geometry import shape
from scipy.ndimage import uniform_filter, binary_dilation, mean as ndi_mean, standard_deviation as ndi_std
from skimage.segmentation import watershed, slic
from skimage.measure import regionprops, regionprops_table
from skimage.feature import peak_local_max
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

import importlib
ground = importlib.import_module('00_ground_truth_helpers')

In [ ]:
# --- Phase 4.1: LiDAR crown detection ---
CHM_TREE_HEIGHT = 1.5             # m, minimum height for a tree
LOCAL_MAX_BASE_RADIUS = 0.5       # m, base of variable window
LOCAL_MAX_HEIGHT_COEF = 0.05      # window_radius = base + coef * height
CROWN_MIN_AREA = 1.0              # m^2
CROWN_MAX_HEIGHT_STD = 3.0        # m, drop merged multi-tree crowns
CROWN_MIN_MAX_HEIGHT = 2        # m, crown max height must exceed this

# --- Phase 4.2: SLIC segmentation at 1m ---
SLIC_N_SEGMENTS = 100_000         # ~10 pixels per segment on 1000x1000 tile
SLIC_COMPACTNESS = 10

# --- Phase 4.3: segment labeling ---
LABEL_TREE_MIN_CROWN_FRACTION = 0.5    # >50% of segment pixels inside a crown
LABEL_NONTREE_CHM_MAX = 0.3            # m
LABEL_NONTREE_NDVI_MAX = 0.15
LABEL_NONTREE_MIN_FRACTION = 0.8       # >80% of segment pixels meet non-tree criteria

# --- Phase 4.7: tree union ---
TREE_PROB_THRESHOLD = 0.7

RF_FEATURE_COLUMNS = [
    'r_mean', 'r_std', 'g_mean', 'g_std', 'b_mean', 'b_std',
    'ndvi_mean', 'ndvi_std', 'savi_mean', 'savi_std',
    'green_local_var_mean', 'ndvi_context_mean',
    'area', 'perimeter', 'eccentricity', 'solidity', 'compactness',
]

# --- I/O grids ---
GRID_SHAPE_1M = (1000, 1000)      # CHM, VI, SLIC, features

In [ ]:
def read_raster(path, out_shape=None, bands=None):
    """Read raster. Returns array, native shape, native band count, and profile."""
    with rasterio.open(path) as src:
        native_shape = (src.height, src.width)
        native_band_count = src.count
        nodata = src.nodata
        profile = src.profile.copy()
        if bands is None:
            bands = list(range(1, native_band_count + 1))
        if out_shape is None:
            arr = src.read(bands)
        else:
            arr = src.read(bands, out_shape=(len(bands), out_shape[0], out_shape[1]))
    arr = arr.astype(np.float32)
    if nodata is not None:
        arr[arr == nodata] = np.nan
    return arr, native_shape, native_band_count, profile


def write_geotiff(array, reference_profile, out_path, dtype, nodata):
    """Write a 2D single-band array as GeoTIFF matching reference_profile."""
    prof = reference_profile.copy()
    prof.update(count=1, dtype=dtype, nodata=nodata, compress='lzw',
                height=array.shape[0], width=array.shape[1])
    with rasterio.open(out_path, 'w', **prof) as dst:
        dst.write(array.astype(dtype), 1)

In [131]:
def detect_tree_tops(chm, min_height, base_radius, height_coef):
    """
    Detect tree tops in a CHM using a variable-radius local-maximum rule.

    Purpose:
        Identify candidate tree top pixels where each peak is the maximum
        within a radius that scales with its own height:
        radius(h) = base_radius + height_coef * h.
        Taller trees claim larger neighborhoods, preventing multiple peaks
        per crown, while shorter trees can still be detected between them.

    Inputs:
        chm         : 2D float array of canopy heights (meters), 1m grid.
        min_height  : minimum peak height to be considered a tree.
        base_radius : base of the variable window (meters == pixels at 1m).
        height_coef : slope of window radius vs. height.
    Outputs:
        tops : (N, 3) float array of [row, col, height] for N detected tops.
    """
    chm_f = np.where(np.isfinite(chm), chm, -np.inf)

    # Candidate peaks: local maxima with 1-pixel neighborhood above min_height
    candidates = peak_local_max(chm_f, min_distance=1, threshold_abs=min_height, exclude_border=False)
    if candidates.size == 0:
        return np.empty((0, 3), dtype=np.float32)

    heights = chm_f[candidates[:, 0], candidates[:, 1]]
    order = np.argsort(-heights)  # tallest first
    candidates = candidates[order]
    heights = heights[order]

    H, W = chm_f.shape
    claimed = np.zeros((H, W), dtype=bool)
    kept = []

    for (r, c), h in zip(candidates, heights):
        if claimed[r, c]:
            continue
        radius = int(np.ceil(base_radius + height_coef * h))
        r0, r1 = max(0, r - radius), min(H, r + radius + 1)
        c0, c1 = max(0, c - radius), min(W, c + radius + 1)
        rs, cs = np.ogrid[r0:r1, c0:c1]
        circle = (rs - r) ** 2 + (cs - c) ** 2 <= radius ** 2
        claimed[r0:r1, c0:c1] |= circle
        kept.append((r, c, h))

    return np.asarray(kept, dtype=np.float32)

In [132]:
def segment_crowns_watershed(chm, tree_tops, min_height):
    """
    Delineate tree crowns via marker-controlled watershed on the inverted CHM.

    Purpose:
        Each tree top is a marker seed. The watershed floods outward from
        each seed on the inverted CHM (peaks become basins), and every
        pixel above min_height is assigned to whichever seed's basin
        reaches it first.

    Inputs:
        chm        : 2D float array of canopy heights (meters).
        tree_tops  : (N, 3) array of [row, col, height] tree tops.
        min_height : mask floor; pixels below this are excluded from any crown.
    Outputs:
        labels : 2D int32 array, 0 = non-crown, > 0 = tree ID.
    """
    H, W = chm.shape
    markers = np.zeros((H, W), dtype=np.int32)
    for i, (r, c, _) in enumerate(tree_tops, start=1):
        markers[int(r), int(c)] = i

    mask = np.isfinite(chm) & (chm > min_height)
    surface = -np.where(np.isfinite(chm), chm, 0)
    labels = watershed(surface, markers=markers, mask=mask)
    return labels.astype(np.int32)

In [133]:
def build_crown_records(labels, chm, tree_tops, transform, crs):
    """
    Build per-crown attribute records with polygons, peak points, and stats.

    Purpose:
        Convert the watershed label raster into crown polygons and per-crown
        attributes (area, height stats, edge-clipping), and apply the
        high-confidence filter for training use.

    Inputs:
        labels     : 2D int32 crown label raster.
        chm        : 2D float CHM used for height stats.
        tree_tops  : (N, 3) [row, col, height] used as canonical peak locations.
        transform  : rasterio Affine for the label raster.
        crs        : CRS of the label raster.
    Outputs:
        crowns_gdf : GeoDataFrame of crown polygons with attributes.
        peaks_gdf  : GeoDataFrame of tree top points with attributes.
    """
    H, W = labels.shape

    # Map tree_id -> peak (row, col, height)
    peaks_by_id = {}
    for i, (r, c, h) in enumerate(tree_tops, start=1):
        peaks_by_id[i] = (float(r), float(c), float(h))

    # Polygonize label raster (skip label 0)
    poly_by_id = {}
    for geom, val in shapes(labels, mask=(labels > 0), transform=transform):
        tid = int(val)
        poly_by_id.setdefault(tid, []).append(shape(geom))
    # Merge multi-part per tid
    from shapely.ops import unary_union
    poly_by_id = {tid: (parts[0] if len(parts) == 1 else unary_union(parts))
                  for tid, parts in poly_by_id.items()}

    # Region properties for stats
    props = regionprops(labels, intensity_image=np.nan_to_num(chm, nan=0.0))
    crown_rows = []
    peak_rows = []

    for p in props:
        tid = int(p.label)
        if tid not in poly_by_id or tid not in peaks_by_id:
            continue
        geom = poly_by_id[tid]
        coords = p.coords
        chm_vals = chm[coords[:, 0], coords[:, 1]]
        chm_vals = chm_vals[np.isfinite(chm_vals)]
        max_h = float(np.max(chm_vals)) if chm_vals.size else 0.0
        mean_h = float(np.mean(chm_vals)) if chm_vals.size else 0.0
        h_std = float(np.std(chm_vals)) if chm_vals.size else 0.0
        minr, minc, maxr, maxc = p.bbox
        edge_clipped = bool(minr == 0 or minc == 0 or maxr == H or maxc == W)
        area_m2 = float(geom.area)

        filter_passed = bool(
            (area_m2 >= CROWN_MIN_AREA)
            and (not edge_clipped)
            and (h_std <= CROWN_MAX_HEIGHT_STD)
            and (max_h >= CROWN_MIN_MAX_HEIGHT)
        )

        crown_rows.append({
            'tree_id': tid,
            'area_m2': area_m2,
            'max_height': max_h,
            'mean_height': mean_h,
            'height_std': h_std,
            'edge_clipped': edge_clipped,
            'filter_passed': filter_passed,
            'geometry': geom,
        })

        pr, pc, ph = peaks_by_id[tid]
        px, py = rasterio.transform.xy(transform, pr, pc)
        peak_rows.append({
            'tree_id': tid,
            'height': ph,
            'filter_passed': filter_passed,
            'geometry': gpd.points_from_xy([px], [py])[0],
        })

    crowns_gdf = gpd.GeoDataFrame(crown_rows, crs=crs)
    peaks_gdf = gpd.GeoDataFrame(peak_rows, crs=crs)
    return crowns_gdf, peaks_gdf

In [ ]:
def run_lidar_crowns_per_tile(tile_id):
    """
    Execute Phase 4.1 for one tile: local maxima -> watershed -> filter,
    then write crown polygons and peak points as GeoPackage files.

    Inputs:
        tile_id : tile identifier string
    Outputs:
        crowns_gdf : GeoDataFrame of crowns (also written to disk)
        peaks_gdf  : GeoDataFrame of peaks (also written to disk)
        labels     : 2D int32 crown label raster (for downstream use)
        chm_profile: rasterio profile of the CHM raster
        chm        : 2D CHM array
    """
    _, _, _, chm_path = ground.build_paths(tile_id)
    print(f"\n[4.1] Tile {tile_id}")
    print(f"CHM: {chm_path.name}")

    chm_arr, chm_native, _, chm_profile = read_raster(chm_path, out_shape=GRID_SHAPE_1M, bands=[1])
    chm = chm_arr[0]
    print(f"CHM native {chm_native}, read {chm.shape}")

    tops = detect_tree_tops(chm, CHM_TREE_HEIGHT, LOCAL_MAX_BASE_RADIUS, LOCAL_MAX_HEIGHT_COEF)

    print(f"detected tree tops: {tops.shape[0]}")

    labels = segment_crowns_watershed(chm, tops, CHM_TREE_HEIGHT)
    n_crowns = int(labels.max())
    print(f"watershed crowns : {n_crowns}")

    crowns_gdf, peaks_gdf = build_crown_records(labels, chm, tops, transform=chm_profile['transform'], crs=chm_profile['crs'])
    n_kept = int(crowns_gdf['filter_passed'].sum())
    print(f"crowns kept (filter_passed): {n_kept} / {len(crowns_gdf)}")

    peaks_path = ground.SAVE_DIR / f"tree_crown_peaks_{tile_id}_{ground.YEAR}.gpkg"
    outlines_path = ground.SAVE_DIR / f"tree_crown_outlines_{tile_id}_{ground.YEAR}.gpkg"
    peaks_gdf.to_file(peaks_path, driver='GPKG')
    crowns_gdf.to_file(outlines_path, driver='GPKG')
    print(f"wrote {peaks_path.name}")
    print(f"wrote {outlines_path.name}")

    return crowns_gdf, peaks_gdf, labels, chm_profile, chm

In [135]:
# %% Cell 8 — Phase 4.2: SLIC segmentation on 1m RGB
def load_rgb_1m(rgb_path):
    """Read RGB downsampled to 1m grid; return (3, H, W) float and profile at 1m grid."""
    rgb_arr, rgb_native, _, _ = read_raster(rgb_path, out_shape=GRID_SHAPE_1M, bands=[1, 2, 3])
    return rgb_arr, rgb_native


def slic_segment(rgb_1m, n_segments, compactness):
    """
    Run SLIC superpixel segmentation on a 1m RGB image.

    Purpose:
        Partition the 1m RGB into ~n_segments compact spatial objects that
        loosely follow color boundaries. Each superpixel is one OBIA unit
        for downstream feature extraction and classification.

    Inputs:
        rgb_1m      : (3, H, W) float array of RGB at 1m grid.
        n_segments  : target number of superpixels.
        compactness : SLIC compactness parameter (higher = more square).
    Outputs:
        segments : 2D int32 array of segment IDs (0 = nodata).
    """
    img = np.transpose(rgb_1m, (1, 2, 0)).astype(np.float32) / 255.0
    nan_mask = ~np.isfinite(img).all(axis=2)
    img = np.nan_to_num(img, nan=0.0)
    segments = slic(img, n_segments=n_segments, compactness=compactness,
                    start_label=1, channel_axis=2)
    segments = segments.astype(np.int32)
    segments[nan_mask] = 0
    return segments

In [136]:
# %% Cell 9 — Phase 4.3: label training segments via LiDAR crowns + CHM/NDVI
def label_segments(segments, crowns_gdf, chm, ndvi, transform):
    """
    Assign 'tree', 'nontree', or 'ambiguous' label to each SLIC segment.

    Purpose:
        Use LiDAR-derived crown polygons (filter_passed only) and CHM/NDVI
        thresholds to build clean training labels for the RGB tree classifier.
        Segments strongly overlapping filter_passed crowns become 'tree';
        segments with predominantly low CHM AND low NDVI become 'nontree';
        the rest are 'ambiguous' and excluded from training.

    Inputs:
        segments   : 2D int32 SLIC label raster on 1m grid.
        crowns_gdf : GeoDataFrame of crown polygons (uses filter_passed==True).
        chm        : 2D float CHM at 1m grid.
        ndvi       : 2D float NDVI at 1m grid.
        transform  : rasterio Affine for the 1m grid.
    Outputs:
        label_map : dict {segment_id: 'tree' | 'nontree' | 'ambiguous'}.
    """
    H, W = segments.shape

    # Rasterize high-confidence crowns to 1m
    kept = crowns_gdf[crowns_gdf['filter_passed']]
    if len(kept) > 0:
        crown_mask = rasterize(
            ((geom, 1) for geom in kept.geometry),
            out_shape=(H, W),
            transform=transform,
            fill=0, dtype='uint8',
        ).astype(bool)
    else:
        crown_mask = np.zeros((H, W), dtype=bool)

    # Non-tree per-pixel condition
    chm_f = np.where(np.isfinite(chm), chm, 999)
    ndvi_f = np.where(np.isfinite(ndvi), ndvi, 999)
    nontree_pixel = (chm_f < LABEL_NONTREE_CHM_MAX) & (ndvi_f < LABEL_NONTREE_NDVI_MAX)

    valid = segments > 0
    seg_flat = segments.ravel()
    valid_flat = valid.ravel()
    n_seg = int(segments.max()) + 1

    total = np.bincount(seg_flat, weights=valid_flat.astype(np.float64), minlength=n_seg)
    tree_pix = np.bincount(seg_flat,
                           weights=(crown_mask & valid).ravel().astype(np.float64),
                           minlength=n_seg)
    nontree_pix = np.bincount(seg_flat,
                              weights=(nontree_pixel & valid).ravel().astype(np.float64),
                              minlength=n_seg)

    label_map = {}
    for sid in range(1, n_seg):
        n = total[sid]
        if n == 0:
            continue
        tree_frac = tree_pix[sid] / n
        nontree_frac = nontree_pix[sid] / n
        if tree_frac > LABEL_TREE_MIN_CROWN_FRACTION:
            label_map[sid] = 'tree'
        elif nontree_frac > LABEL_NONTREE_MIN_FRACTION:
            label_map[sid] = 'nontree'
        else:
            label_map[sid] = 'ambiguous'
    return label_map

In [137]:
# %% Cell 10 — Phase 4.4: per-segment feature extraction (transferable set)
def extract_segment_features(segments, rgb_1m, ndvi, savi):
    """
    Compute per-segment features from RGB + NDVI + SAVI (transferable to NAIP).

    Features per segment:
        Spectral : mean and std of R, G, B, NDVI, SAVI
        Texture  : mean of 3x3 local variance of the green channel
        Shape    : area, perimeter, eccentricity, solidity, compactness
        Context  : mean of 5x5 local NDVI

    Inputs:
        segments : 2D int32 SLIC label raster.
        rgb_1m   : (3, H, W) RGB at 1m.
        ndvi     : 2D float NDVI at 1m.
        savi     : 2D float SAVI at 1m.
    Outputs:
        df : pandas.DataFrame indexed by segment_id with feature columns.
    """
    seg_ids = np.unique(segments)
    seg_ids = seg_ids[seg_ids > 0]

    r_ch = np.nan_to_num(rgb_1m[0], nan=0.0)
    g_ch = np.nan_to_num(rgb_1m[1], nan=0.0)
    b_ch = np.nan_to_num(rgb_1m[2], nan=0.0)
    ndvi_f = np.nan_to_num(ndvi, nan=0.0)
    savi_f = np.nan_to_num(savi, nan=0.0)

    feats = {'segment_id': seg_ids}

    for name, arr in [('r', r_ch), ('g', g_ch), ('b', b_ch),
                      ('ndvi', ndvi_f), ('savi', savi_f)]:
        feats[f'{name}_mean'] = ndi_mean(arr, labels=segments, index=seg_ids)
        feats[f'{name}_std']  = ndi_std(arr, labels=segments, index=seg_ids)

    # Texture: local variance of green (3x3)
    local_var_g = uniform_filter(g_ch ** 2, size=3) - uniform_filter(g_ch, size=3) ** 2
    feats['green_local_var_mean'] = ndi_mean(local_var_g, labels=segments, index=seg_ids)

    # Context: local NDVI mean (5x5)
    local_ndvi = uniform_filter(ndvi_f, size=5)
    feats['ndvi_context_mean'] = ndi_mean(local_ndvi, labels=segments, index=seg_ids)

    # Shape via regionprops
    props = regionprops_table(
        segments,
        properties=('label', 'area', 'perimeter', 'eccentricity', 'solidity'),
    )
    shape_df = pd.DataFrame(props).set_index('label').reindex(seg_ids)
    shape_df['compactness'] = 4 * np.pi * shape_df['area'] / (shape_df['perimeter'] ** 2 + 1e-6)
    for col in ['area', 'perimeter', 'eccentricity', 'solidity', 'compactness']:
        feats[col] = shape_df[col].values

    df = pd.DataFrame(feats).set_index('segment_id')
    return df

In [ ]:
# %% Cell 11 — Run Phase 4.2–4.4 per tile and pool labeled segments
def process_tile_rgb(tile_id, crowns_gdf, chm_profile, chm_arr):
    """
    Run SLIC + labeling + feature extraction on one tile.

    Inputs:
        tile_id     : tile string
        crowns_gdf  : LiDAR crown polygons from Phase 4.1 (with filter_passed)
        chm_profile : rasterio profile of the 1m grid (used as reference)
        chm_arr     : 2D CHM at 1m grid
    Outputs:
        features_df   : pandas.DataFrame of per-segment features
        label_map     : dict segment_id -> label ('tree' | 'nontree' | 'ambiguous')
        segments      : 2D int32 SLIC raster
        rgb_1m        : (3, H, W) RGB at 1m
        ndvi, savi    : 2D float arrays at 1m
    """
    rgb_path, ndvi_path, savi_path, _ = ground.build_paths(tile_id)
    print(f"\n[4.2-4.4] Tile {tile_id}")

    rgb_1m, rgb_native = load_rgb_1m(rgb_path)
    print(f"RGB native {rgb_native}, read {rgb_1m.shape[1:]}")

    ndvi_arr, ndvi_native, _, _ = read_raster(ndvi_path, out_shape=GRID_SHAPE_1M, bands=[1])
    savi_arr, savi_native, _, _ = read_raster(savi_path, out_shape=GRID_SHAPE_1M, bands=[1])
    ndvi = ndvi_arr[0]
    savi = savi_arr[0]

    segments = slic_segment(rgb_1m, SLIC_N_SEGMENTS, SLIC_COMPACTNESS)
    n_seg = int(segments.max())
    print(f"SLIC segments   : {n_seg}")

    label_map = label_segments(segments, crowns_gdf, chm_arr, ndvi, transform=chm_profile['transform'])
    counts = pd.Series(list(label_map.values())).value_counts().to_dict()
    print(f"segment labels  : {counts}")

    features_df = extract_segment_features(segments, rgb_1m, ndvi, savi)
    features_df['label'] = features_df.index.map(label_map)
    features_df['tile_id'] = tile_id
    print(f"feature rows    : {len(features_df)}")

    return features_df, label_map, segments, rgb_1m, ndvi, savi

In [ ]:
# %% Cell 12 — Phase 4.5: leave-one-tile-out CV + final RF training
def loo_cross_validate(all_features, RF_FEATURE_COLUMNS):
    """
    Leave-one-tile-out cross-validation for the RGB tree detector.

    Purpose:
        Report an honest cross-tile generalization estimate by training on
        n-1 tiles and testing on the held-out tile, rotated across all tiles.
        Uses only 'tree' and 'nontree' labeled segments; 'ambiguous' excluded.

    Inputs:
        all_features : pooled per-segment DataFrame with 'label' and 'tile_id'
        RF_FEATURE_COLUMNS : list of feature column names used for training
    Outputs:
        None (prints per-fold classification reports and pooled confusion matrix)
    """
    train_df = all_features[all_features['label'].isin(['tree', 'nontree'])].copy()
    y_true_all, y_pred_all = [], []
    for tile_id in train_df['tile_id'].unique():
        train = train_df[train_df['tile_id'] != tile_id]
        test = train_df[train_df['tile_id'] == tile_id]
        Xtr, ytr = train[RF_FEATURE_COLUMNS].values, train['label'].values
        Xte, yte = test[RF_FEATURE_COLUMNS].values, test['label'].values
        clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42, class_weight='balanced')
        clf.fit(Xtr, ytr)
        yhat = clf.predict(Xte)
        print(f"\n  Fold: held-out tile {tile_id}  (n_train={len(train)}, n_test={len(test)})")
        print(classification_report(yte, yhat, zero_division=0))
        y_true_all.extend(yte)
        y_pred_all.extend(yhat)
    print("\n=== Pooled LOO confusion matrix ===")
    print(confusion_matrix(y_true_all, y_pred_all, labels=['nontree', 'tree']))


def train_final_model(all_features, RF_FEATURE_COLUMNS):
    """
    Train the final RF tree detector on all tree/nontree segments.

    Inputs:
        all_features : pooled per-segment DataFrame with 'label' column
        RF_FEATURE_COLUMNS : list of feature column names
    Outputs:
        clf : trained RandomForestClassifier (also saved as pickle)
    """
    train_df = all_features[all_features['label'].isin(['tree', 'nontree'])]
    X, y = train_df[RF_FEATURE_COLUMNS].values, train_df['label'].values
    clf = RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42, class_weight='balanced')
    clf.fit(X, y)
    model_path = ground.SAVE_DIR / f"rgb_tree_detector_{ground.YEAR}.pkl"
    with open(model_path, 'wb') as f:
        pickle.dump({'model': clf, 'RF_FEATURE_COLUMNS': RF_FEATURE_COLUMNS}, f)
    print(f"\n  saved model: {model_path.name}")
    print(f"feature importances:")
    for name, imp in sorted(zip(RF_FEATURE_COLUMNS, clf.feature_importances_), key=lambda t: -t[1]):
        print(f"  {name:25s} {imp:.4f}")
    return clf

In [ ]:
# %% Cell 13 — Phase 4.6–4.8: predict, union tree mask, and write confidence
def tree_outputs_per_tile(tile_id, clf, features_df, segments, chm_arr, chm_profile, RF_FEATURE_COLUMNS):
    """
    Apply the trained RF, union with CHM > 2m, and write final tree mask and
    tree confidence rasters for one tile.

    Purpose:
        Phase 4.6 : predict tree probability per SLIC segment.
        Phase 4.7 : tree mask = (CHM > 2m) UNION (RGB prob > TREE_PROB_THRESHOLD).
        Phase 4.8 : confidence: High = both sources agree; Medium = CHM-only or
                    RGB prob > 0.85; Low = RGB-only with prob in 0.7..0.85.

    Inputs:
        tile_id      : tile string
        clf          : trained RandomForestClassifier
        features_df  : per-segment features for this tile
        segments     : 2D int32 SLIC raster for this tile
        chm_arr      : 2D CHM at 1m for this tile
        chm_profile  : rasterio profile of the 1m grid
        RF_FEATURE_COLUMNS : list of feature column names
    Outputs:
        None (writes tree mask, confidence raster, and segment probability raster)
    """
    tile_feats = features_df[features_df['tile_id'] == tile_id].copy()
    X = tile_feats[RF_FEATURE_COLUMNS].values
    proba = clf.predict_proba(X)
    tree_idx = list(clf.classes_).index('tree')
    tile_feats['tree_prob'] = proba[:, tree_idx]

    # Rasterize probability back to segments
    prob_map = np.zeros(segments.shape, dtype=np.float32)
    prob_map[:] = np.nan
    prob_lookup = tile_feats['tree_prob'].to_dict()
    seg_ids = np.unique(segments)
    seg_ids = seg_ids[seg_ids > 0]
    lookup_arr = np.full(int(seg_ids.max()) + 1, np.nan, dtype=np.float32)
    for sid, p in prob_lookup.items():
        lookup_arr[int(sid)] = p
    valid = segments > 0
    prob_map[valid] = lookup_arr[segments[valid]]

    # Union tree mask
    chm_tree = np.isfinite(chm_arr) & (chm_arr > CHM_TREE_HEIGHT)
    rgb_tree = np.isfinite(prob_map) & (prob_map > TREE_PROB_THRESHOLD)
    tree_mask = (chm_tree | rgb_tree).astype(np.uint8)

    # Confidence stratification (1 = low, 2 = medium, 3 = high, 0 = not tree)
    confidence = np.zeros(segments.shape, dtype=np.uint8)
    both = chm_tree & rgb_tree
    chm_only = chm_tree & ~rgb_tree
    rgb_high = rgb_tree & (prob_map > 0.85) & ~chm_tree
    rgb_low = rgb_tree & (prob_map <= 0.85) & ~chm_tree
    confidence[both] = 3
    confidence[chm_only | rgb_high] = np.maximum(confidence[chm_only | rgb_high], 2)
    confidence[rgb_low] = np.maximum(confidence[rgb_low], 1)

    # Write outputs
    mask_path = ground.SAVE_DIR / f"tree_mask_{tile_id}_{ground.YEAR}.tif"
    conf_path = ground.SAVE_DIR / f"tree_confidence_{tile_id}_{ground.YEAR}.tif"
    prob_path = ground.SAVE_DIR / f"tree_segment_prob_{tile_id}_{ground.YEAR}.tif"
    write_geotiff(tree_mask, chm_profile, mask_path, dtype='uint8', nodata=255)
    write_geotiff(confidence, chm_profile, conf_path, dtype='uint8', nodata=255)
    write_geotiff(np.where(np.isfinite(prob_map), prob_map, -1).astype(np.float32), chm_profile, prob_path, dtype='float32', nodata=-1)

    n_tree = int(tree_mask.sum())
    n_valid = int((tree_mask != 255).sum())
    print(f"[{tile_id}] tree pixels: {n_tree:,} / {n_valid:,} ({n_tree / n_valid * 100:.2f}%)")
    print(f"[{tile_id}] wrote {mask_path.name}, {conf_path.name}, {prob_path.name}")

In [ ]:
print(f"=== Phase 4: tree detection ({ground.SITE_ID} {ground.YEAR}, {len(ground.TILE_IDS)} tiles) ===")

# 4.1: LiDAR crowns per tile
tile_state = {}
for tid in ground.TILE_IDS:
    crowns_gdf, peaks_gdf, labels, chm_profile, chm = run_lidar_crowns_per_tile(tid)
    tile_state[tid] = dict(crowns=crowns_gdf, chm_profile=chm_profile, chm=chm)

=== Phase 4: tree detection (SRER 2022, 3 tiles) ===

[4.1] Tile 515000_3530000
CHM: NEON_D14_SRER_DP3_515000_3530000_CHM.tif
CHM native (1000, 1000), read (1000, 1000)
detected tree tops: 39135
watershed crowns : 39135
crowns kept (filter_passed): 17223 / 39135
wrote tree_crown_peaks_515000_3530000_2022.gpkg
wrote tree_crown_outlines_515000_3530000_2022.gpkg

[4.1] Tile 515000_3531000
CHM: NEON_D14_SRER_DP3_515000_3531000_CHM.tif
CHM native (1000, 1000), read (1000, 1000)
detected tree tops: 27235
watershed crowns : 27235
crowns kept (filter_passed): 12005 / 27235
wrote tree_crown_peaks_515000_3531000_2022.gpkg
wrote tree_crown_outlines_515000_3531000_2022.gpkg

[4.1] Tile 511000_3528000
CHM: NEON_D14_SRER_DP3_511000_3528000_CHM.tif
CHM native (1000, 1000), read (1000, 1000)
detected tree tops: 13947
watershed crowns : 13947
crowns kept (filter_passed): 10152 / 13947
wrote tree_crown_peaks_511000_3528000_2022.gpkg
wrote tree_crown_outlines_511000_3528000_2022.gpkg


In [ ]:
# 4.2-4.4: SLIC + label + features per tile
all_features = []
seg_by_tile = {}
for tid in ground.TILE_IDS:
    feats, label_map, segments, rgb_1m, ndvi, savi = process_tile_rgb(tid, tile_state[tid]['crowns'],tile_state[tid]['chm_profile'], tile_state[tid]['chm'])
    all_features.append(feats)
    seg_by_tile[tid] = segments
all_features = pd.concat(all_features, axis=0)


[4.2-4.4] Tile 515000_3530000
RGB native (10000, 10000), read (1000, 1000)
SLIC segments   : 92596
segment labels  : {'ambiguous': 75891, 'tree': 15990, 'nontree': 715}


/projectnb/modislc/fache/.conda/envs/LCSC/lib/python3.11/site-packages/scipy/ndimage/_measurements.py:647: RuntimeWarning: invalid value encountered in divide
  means = sums / counts


feature rows    : 92596

[4.2-4.4] Tile 515000_3531000
RGB native (10000, 10000), read (1000, 1000)
SLIC segments   : 92521
segment labels  : {'ambiguous': 80260, 'tree': 10415, 'nontree': 1846}


/projectnb/modislc/fache/.conda/envs/LCSC/lib/python3.11/site-packages/scipy/ndimage/_measurements.py:647: RuntimeWarning: invalid value encountered in divide
  means = sums / counts


feature rows    : 92521

[4.2-4.4] Tile 511000_3528000
RGB native (10000, 10000), read (1000, 1000)
SLIC segments   : 98609
segment labels  : {'ambiguous': 83589, 'tree': 13482, 'nontree': 1538}


/projectnb/modislc/fache/.conda/envs/LCSC/lib/python3.11/site-packages/scipy/ndimage/_measurements.py:647: RuntimeWarning: invalid value encountered in divide
  means = sums / counts


feature rows    : 98609


In [123]:
all_features.head()

,r_mean,r_std,g_mean,g_std,b_mean,b_std,ndvi_mean,ndvi_std,savi_mean,savi_std,green_local_var_mean,ndvi_context_mean,area,perimeter,eccentricity,solidity,compactness,label,tile_id
segment_id,,,,,,,,,,,,,,,,,,,
1,169.777778,31.832121,156.222222,30.176436,142.555556,28.103425,0.276528,0.101145,0.174691,0.050894,1001.673394,0.319875,9.0,8.828427,0.786953,1.000000,1.451061,ambiguous,515000_3530000
2,158.200000,11.070682,149.200000,9.846827,127.800000,11.600000,0.512882,0.055423,0.277938,0.020241,711.569531,0.412950,5.0,5.207107,0.816497,1.000000,2.317325,ambiguous,515000_3530000
3,212.400000,27.026654,199.200000,25.051148,181.800000,29.318254,0.279282,0.118985,0.191448,0.065831,1410.572656,0.314554,10.0,8.000000,0.802571,0.833333,1.963495,ambiguous,515000_3530000
4,146.500000,23.314874,136.833333,20.111495,123.500000,16.680827,0.312527,0.072835,0.196585,0.024773,1557.764648,0.245917,6.0,6.414214,0.870388,1.000000,1.832628,ambiguous,515000_3530000
5,224.888889,8.211705,208.666667,8.299933,189.555556,9.967850,0.139569,0.054725,0.112854,0.038694,659.750000,0.179604,9.0,9.656854,0.902671,1.000000,1.212777,ambiguous,515000_3530000


In [ ]:
# 4.5: LOO cross-validation + final model
print("\n=== 4.5 LOO cross-validation ===")
loo_cross_validate(all_features, RF_FEATURE_COLUMNS)
print("\n=== 4.5 Final model training on pooled labels ===")
clf = train_final_model(all_features, RF_FEATURE_COLUMNS)


=== 4.5 LOO cross-validation ===

  Fold: held-out tile 515000_3530000  (n_train=27281, n_test=16705)
              precision    recall  f1-score   support

     nontree       0.99      1.00      0.99       715
        tree       1.00      1.00      1.00     15990

    accuracy                           1.00     16705
   macro avg       0.99      1.00      1.00     16705
weighted avg       1.00      1.00      1.00     16705


  Fold: held-out tile 515000_3531000  (n_train=31725, n_test=12261)
              precision    recall  f1-score   support

     nontree       0.99      1.00      0.99      1846
        tree       1.00      1.00      1.00     10415

    accuracy                           1.00     12261
   macro avg       1.00      1.00      1.00     12261
weighted avg       1.00      1.00      1.00     12261


  Fold: held-out tile 511000_3528000  (n_train=28966, n_test=15020)
              precision    recall  f1-score   support

     nontree       1.00      1.00      1.00      1

In [ ]:
# 4.6-4.8: predict, union, write tree mask + confidence
print("\n=== 4.6-4.8 Predict + union + confidence ===")
for tid in ground.TILE_IDS:
    tree_outputs_per_tile(tid, clf, all_features, seg_by_tile[tid], tile_state[tid]['chm'], tile_state[tid]['chm_profile'], RF_FEATURE_COLUMNS)


=== 4.6-4.8 Predict + union + confidence ===
[515000_3530000] tree pixels: 965,776 / 1,000,000 (96.58%)
[515000_3530000] wrote tree_mask_515000_3530000_2022.tif, tree_confidence_515000_3530000_2022.tif, tree_segment_prob_515000_3530000_2022.tif
[515000_3531000] tree pixels: 936,192 / 1,000,000 (93.62%)
[515000_3531000] wrote tree_mask_515000_3531000_2022.tif, tree_confidence_515000_3531000_2022.tif, tree_segment_prob_515000_3531000_2022.tif
[511000_3528000] tree pixels: 971,049 / 1,000,000 (97.10%)
[511000_3528000] wrote tree_mask_511000_3528000_2022.tif, tree_confidence_511000_3528000_2022.tif, tree_segment_prob_511000_3528000_2022.tif
